# A rich-quench-lean (RQL) combustor sector and its indirect-noise instability

This example takes a single sector of an annular RQL combustor, solves its **reacting,
multi-stream mean flow** on a branched network, and shows that the mean flow carries an
**indirect (entropy-driven) combustion instability**.  The unstable mode is *acoustic in energy*
but *entropy in timing*: the flame's unsteady heat release sheds a hot spot (an entropy
fluctuation) that convects to the choked nozzle guide vane, is turned back into sound there
(Marble–Candel indirect noise), and returns to the flame — a feedback loop that runs **through
the entropy wave**.  Dropping the entropy wave removes the instability.

The network is the branched RQL architecture: compressor-discharge air enters a diffuser and
**splits** between the swirler (primary) dome, the inner annulus, and the outer annulus; the
annuli meter cooling and dilution air back into the liner through orifices; the rich primary
products are quenched to lean and pass a **choked nozzle guide vane**.  The air splits are not
prescribed — they are solved from the passage and orifice areas.

In [ ]:
from IPython.display import display

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import nefes
from nefes.chem import equivalence_ratio_mixture
from nefes.elements import heat_release_response, n_tau_lowpass
from nefes.perturbation import PerturbationBC
from nefes.plotting import use_nefes_theme, COLORWAY

use_nefes_theme()

CASE = "rql_combustor.yaml"
# Same mole composition as inlet-30 in the case file (dry air with Ar and CO2).
AIR = {"O2": 0.2095, "N2": 0.7808, "Ar": 0.0093, "CO2": 0.0004}
FUEL = {"Jet-A(L)": 1.0}

## 1. The reacting mean flow

Load the dimensioned case and solve the steady mean flow.  The gas model is chemical
equilibrium with a Jet-A(L) fuel; the condensed-soot phase C(gr) is kept in the species set so
the rich primary is well posed.  The solve converges on the default continuation.

In [ ]:
net = nefes.load_case(CASE)
sol = net.solve()
assert sol.converged, (sol.residual_norm, sol.print_residuals())
sol.verify()  # mass balance, subsonic Mach, choking consistency
xbar = sol.x  # the converged mean flow; the flame response does not change it, so it is reused below

# Stoichiometric Jet-A / air mass ratio from the case air composition.
_st = equivalence_ratio_mixture(FUEL, AIR, 1.0, basis="mass", species_set=net.gas.species_set)
FAR_STOICH = _st["Jet-A(L)"] / (1.0 - _st["Jet-A(L)"])

mdot = sol.field("mdot")


def md(a, b):
    """Mean mass flow on the edge between two named elements [kg/s]."""
    return abs(mdot[net.edge_between(net.element_index(a), net.element_index(b))])


W_air = md("inlet-30", "diffuser")
W_fuel = md("fuel-in", "sw-dump") - md("sw-duct", "fuel-in")  # fuel added at the injector
swirler = md("dome-split", "sw-in")
cool1 = md("cool-inner-1", "mix-cool-1") + md("cool-outer-1", "mix-cool-1")
dilution = md("dil-inner", "mix-dil") + md("dil-outer", "mix-dil")
cool2 = md("cool-inner-2", "mix-cool-2") + md("cool-outer-2", "mix-cool-2")

print("Sector feed")
print(f"  {'quantity':<12}  {'value':>10}")
print(f"  {'air':<12}  {W_air:10.3f} kg/s")
print(f"  {'fuel':<12}  {W_fuel:10.4f} kg/s")
print(f"  {'overall phi':<12}  {W_fuel / (FAR_STOICH * W_air):10.3f}")
print()
print("Air splits")
print(f"  {'branch':<20}  {'mdot [kg/s]':>11}  {'share':>7}")
for name, m in [
    ("swirler / primary", swirler),
    ("cooling-1", cool1),
    ("dilution / quench", dilution),
    ("cooling-2", cool2),
]:
    print(f"  {name:<20}  {m:11.4f}  {100 * m / W_air:6.1f}%")

The air distribution and the resulting equivalence-ratio staging are the RQL signature: a rich,
oxygen-depleted primary; a small cooling bleed that keeps the gas rich; then a large dilution
jet that quenches it through stoichiometric to lean.

In [ ]:
def phi_at(cum_air):
    return W_fuel / (FAR_STOICH * cum_air)


print("Equivalence-ratio staging")
print(f"  {'station':<18}  {'phi':>6}")
for name, air in [
    ("primary (rich)", swirler),
    ("after cooling-1", swirler + cool1),
    ("after dilution", swirler + cool1 + dilution),
    ("exit", W_air),
]:
    print(f"  {name:<18}  {phi_at(air):6.3f}")

In [ ]:
net.plot(
    solution=sol, color_by="T", width_by="mdot", colorscale="Inferno", title="RQL sector: solved static temperature [K]"
).show()

In [ ]:
def T_at(a, b):
    return sol.field("T")[net.edge_between(net.element_index(a), net.element_index(b))]


path = [
    ("primary", ("cc-duct-1", "flame"), swirler),
    ("rich burn", ("flame", "cc-duct-2"), swirler),
    ("post cooling-1", ("cc-duct-3", "mix-dil"), swirler + cool1),
    ("post dilution", ("mix-dil", "cc-duct-4"), swirler + cool1 + dilution),
    ("exit (NGV)", ("nozzle", "ngv"), W_air),
]
labels = [p[0] for p in path]
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_scatter(
    x=labels,
    y=[T_at(*p[1]) for p in path],
    name="temperature",
    mode="lines+markers",
    line=dict(color=COLORWAY[4], width=3),
)
fig.add_scatter(
    x=labels,
    y=[phi_at(p[2]) for p in path],
    name="equivalence ratio",
    mode="lines+markers",
    line=dict(color=COLORWAY[0], width=3, dash="dash"),
    secondary_y=True,
)
fig.add_hline(y=1.0, line=dict(color="gray", dash="dot"), secondary_y=True)
fig.update_yaxes(title_text="static temperature [K]", secondary_y=False)
fig.update_yaxes(title_text="equivalence ratio", secondary_y=True)
fig.update_layout(title="RQL staging: rich primary, quench through stoichiometric, lean exit")
fig.show()

## 2. The flame response

Give the flame a heat-release response: a low-pass n-tau flame transfer function reading the
velocity fluctuation just upstream of the flame.  Its unsteady heat release makes both sound and
entropy spots — no separate entropy source is needed.  The lag is later swept to expose what
sets the mode frequency.

In [ ]:
N_FTF, TAU, FC = 2.0, 5.0e-3, 300.0  # gain, time lag [s], roll-off [Hz]

# Attach the response once; the mean flow ignores it, so xbar above stays valid throughout.
flame = net.element_index("flame")
ref_edge = net.edge_between(net.element_index("cc-duct-1"), flame)
net.set_dynamic_source(flame, heat_release_response(n_tau_lowpass(N_FTF, TAU, FC), ref_edge=ref_edge))


def modes(network, isentropic, band=(60.0, 130.0), gb=(-160.0, 120.0)):
    s = network.solve(x0=xbar)
    return s.eigenmodes(freq_band=band, growth_band=gb, isentropic=isentropic)

## 3. Test 1 — drop the entropy wave, and the instability disappears

Run the eigensolver twice on the *same* mean flow and operator, changing only whether the
convected entropy wave is carried.  `isentropic=True` pins the entropy fluctuation to zero in the
ducts, so the flame's hot spot cannot reach the nozzle and the indirect path is cut;
`isentropic=False` retains it.  The ~90 Hz mode is **stable** as a pure acoustic problem and is
driven **unstable** only when the entropy wave is retained — the mark of a genuinely indirect
mode.

In [ ]:
full = modes(net, isentropic=False)
iso = modes(net, isentropic=True)

print("entropy ON (full)")
display(full)
print("entropy OFF (isentropic)")
display(iso)

iu = int(np.argmax(full.growth_rates))
f0 = full.freqs[iu]

# The eigenvector is acoustic-dominated: the entropy carries the timing, not the energy.
ms = np.abs(full.mode_shape(iu, basis="char"))
print("Mode-shape split (unstable full mode)")
print(
    f"  max|h| / max|f,g| = {ms[:, 2].max() / ms[:, :2].max():.3f}"
    f"   (sound carries the energy; entropy carries the clock)"
)

For completeness: this combustor has **no lightly-damped acoustic cavity mode** in band (the
choked nozzle, orifices and junctions are strongly dissipative), and a genuine *intrinsic* (ITA)
mode does exist near ~100 Hz but stays strongly damped — so the unstable mode is unambiguously
the indirect one, not a cavity or ITA mode.

## 4. What sets the frequency: the feedback-loop period

The indirect loop closes over three legs: the entropy **convects** flame → nozzle in
$\tau_\mathrm{conv}=\int \mathrm{d}x/\overline u$, the reflected sound **returns** nozzle → flame
in $\tau_\mathrm{ac}=\int \mathrm{d}x/(\overline c-\overline u)$, and the flame adds its lag
$\tau$.  The mode sits on the harmonic ladder $f = k/T_\mathrm{loop}$ with
$T_\mathrm{loop}=\tau_\mathrm{conv}+\tau_\mathrm{ac}+\tau$.  Because the convective leg is slow,
this ladder is far below the acoustic one — a low-frequency mode from a short, hot chamber.

In [ ]:
HOT_DUCTS = [("cc-duct-2", 0.05), ("cc-duct-3", 0.07), ("cc-duct-4", 0.06)]  # flame -> NGV ducts (name, length [m])


def loop_legs(network, solution):
    """Convective and acoustic-return times over the hot ducts [s]."""
    tconv = tac = 0.0
    for name, L in HOT_DUCTS:
        ed = solution.edge(network.edges_of(network.element_index(name))[0])
        tconv += L / ed["u"]
        tac += L / (ed["c"] - ed["u"])
    return tconv, tac


tconv, tac = loop_legs(net, sol)
T_loop = tconv + tac + TAU

print("Loop times [ms]")
print(f"  {'tau_conv  flame -> NGV':<28}  {tconv * 1e3:6.2f}")
print(f"  {'tau_ac    NGV -> flame':<28}  {tac * 1e3:6.2f}")
print(f"  {'tau       flame lag':<28}  {TAU * 1e3:6.2f}")
print(f"  {'T_loop':<28}  {T_loop * 1e3:6.2f}")
print()
print("Ladder  k / T_loop")
print("  " + "   ".join(f"k={k}: {k / T_loop:.0f} Hz" for k in (1, 2, 3)))
print(f"  unstable mode at {f0:.0f} Hz is the k=1 loop mode  (1/f = {1e3 / f0:.1f} ms)")

## 5. Test 2 — the frequency slides with the flame lag along k / T_loop

The decisive test.  Sweep the flame lag $\tau$ and track the eigenvalues.  The geometry (and thus
any acoustic mode) is fixed, and $\tau$ enters only through $T_\mathrm{loop}$, so a loop-timed
mode must slide along $f=k/T_\mathrm{loop}(\tau)$ — which a cavity mode (horizontal) or a pure ITA
mode (steeper, $\sim 1/2\tau$) could not do.  The dotted curve is the predicted ladder; the mode
rides it.

In [ ]:
taus = np.linspace(3.0e-3, 7.0e-3, 12)
traj_tau = net.eigenvalue_trajectory(
    "flame.dynamic_source.tau",
    taus,
    freq_band=(50.0, 170.0),
    growth_band=(-200.0, 150.0),
    isentropic=False,
    param_name="flame lag τ [s]",
)
display(traj_tau)

fig = traj_tau.plot_vs_param(title="Eigenvalues vs flame lag τ: the mode rides the loop-period ladder")
fig.add_scatter(
    x=taus,
    y=1.0 / (tconv + tac + taus),
    mode="lines",
    name="1 / T_loop (indirect)",
    line=dict(color="#9aa5b1", dash="dot", width=2),
    row=1,
    col=1,
)
fig.show()

slope = np.polyfit(taus * 1e3, [1.0 / (tconv + tac + t) for t in taus], 1)[0]
print("Frequency sensitivity to flame lag")
print(f"  indirect prediction  df/dτ ≈ {slope:6.1f} Hz/ms")
print(f"  pure ITA reference             ≈ {-0.5 / (TAU ** 2) / 1e3:6.0f} Hz/ms")

## 6. The onset: raise the flame gain, and confirm the nozzle is in the loop

Raising the flame gain feeds the loop; the mode crosses the stability line near gain ≈ 1.8.  And
because the loop closes at the nozzle's entropy-to-sound conversion, making the nozzle
**anechoic** (no conversion) removes the instability entirely — the loop physically runs through
the choked nozzle guide vane.

In [ ]:
gains = np.linspace(0.2, 2.2, 12)
traj_n = net.eigenvalue_trajectory(
    "flame.dynamic_source.n",
    gains,
    freq_band=(60.0, 130.0),
    growth_band=(-200.0, 150.0),
    isentropic=False,
    param_name="flame gain n",
)
display(traj_n)

fig = traj_n.plot_vs_param(title="Eigenvalues vs flame gain n: crossing into instability near n ≈ 1.8")
fig.show()

print("anechoic NGV (entropy-to-sound conversion removed)")
anechoic_net = net.copy()
anechoic_net.set_perturbation_bc(anechoic_net.element_index("ngv"), PerturbationBC.anechoic())
anech = modes(anechoic_net, isentropic=False, band=(60.0, 130.0), gb=(-250.0, 120.0))
display(anech)

## 7. Real-frequency confirmation (Nyquist)

The eigenvalue verdict is corroborated on the real frequency axis, where the convected entropy
wave is handled exactly.  With the entropy wave dropped the flame return ratio does not encircle
the critical point; retaining it, the locus encircles once (one unstable mode).

In [ ]:
freqs = np.linspace(1.0, 700.0, 900)
sol_f = net.solve(x0=xbar)
nq_iso = sol_f.nyquist_stability(freqs, isentropic=True)
nq_full = sol_f.nyquist_stability(freqs, isentropic=False)

print("acoustic only (isentropic)")
display(nq_iso)
print("entropy retained (full)")
display(nq_full)

fig = make_subplots(rows=1, cols=2, subplot_titles=("acoustic only — stable", "entropy retained — unstable"))
for col, r in ((1, nq_iso), (2, nq_full)):
    fig.add_scatter(
        x=r.L.real, y=r.L.imag, mode="lines", line=dict(color=COLORWAY[0]), showlegend=False, row=1, col=col
    )
    fig.add_scatter(
        x=r.L.real,
        y=-r.L.imag,
        mode="lines",
        line=dict(color=COLORWAY[0], dash="dot"),
        showlegend=False,
        row=1,
        col=col,
    )
    fig.add_scatter(
        x=[1.0],
        y=[0.0],
        mode="markers",
        marker=dict(symbol="x", size=12, color=COLORWAY[4]),
        showlegend=False,
        row=1,
        col=col,
    )
fig.update_xaxes(title_text="Re L")
fig.update_yaxes(title_text="Im L", col=1)
fig.update_layout(title="Nyquist locus of the flame return ratio")
fig.show()

## Summary

* A single **RQL sector** is solved as a branched reacting network: the swirler/cooling/dilution
  air **splits are solved**, not prescribed, giving a realistic rich-quench-lean staging (rich,
  oxygen-depleted primary φ ≈ 1.65; quench to lean; realistic turbine-inlet temperature).
* The unstable ~90 Hz mode is a genuine **indirect (entropy-driven) instability**: it vanishes
  when the entropy wave is dropped and when the nozzle's entropy-to-sound conversion is removed,
  and its eigenvector is acoustic-dominated (entropy ≈ 4% of the amplitude).
* Its frequency is set by the **feedback-loop period** $T_\mathrm{loop}=\tau_\mathrm{conv}+
  \tau_\mathrm{ac}+\tau$: it sits on the $k=1$ ladder and **slides with the flame lag** along
  $k/T_\mathrm{loop}$ (the slow convective leg is in the period), which no cavity or purely
  intrinsic mode could do.  The gain trajectory locates the onset near $n\approx1.8$, and Nyquist
  confirms it on the real axis.